In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Population Dynamics Insights: WorldPop-Weighted Custom Boundary Aggregation

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fpopulation_dynamics_insights%2Fnotebooks%2Fworldpop_weighted_custom_boundary_aggregation%2Fpdi_worldpop_weighted_custom_boundary_aggregation.ipynb?utm_source=pdi_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/population_dynamics_insights/notebooks/worldpop_weighted_custom_boundary_aggregation/pdi_worldpop_weighted_custom_boundary_aggregation.ipynb&utm_source=pdi_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/population_dynamics_insights/notebooks/worldpop_weighted_custom_boundary_aggregation/pdi_worldpop_weighted_custom_boundary_aggregation.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

> **⚠️ Important Requirement:** To run the queries in this notebook, your Google Cloud Project must have access to the **US Population Dynamics Insights dataset**. For instructions on how to request and configure access, see [Set up Population Dynamics Insights](https://developers.google.com/maps/documentation/population-dynamics-insights/cloud-setup).

### Overall Goal

This guide demonstrates how to use **Population Dynamics Insights (PDI)** alongside **Google Earth Engine (GEE)** to perform **Custom Boundary Aggregation**: taking native S2 cell (Level 12) embeddings and accurately rolling them up into custom, arbitrary polygons (like drive-time isochrones or sales territories) using a highly precise **Population-Weighted Average**.

**The Scenario:** You want to use PDI's 330-dimensional embeddings as features in an ML model predicting retail store performance. Your target geographic boundaries are 5km radii. Because human activity is rarely spread evenly across physical space, a naive "area-weighted" aggregation (assuming a 50% overlap equals 50% of the signal) will warp your ML features. By performing all calculations directly in BigQuery, we leverage high-resolution raster population data, specifically the[WorldPop USA 2025 Population Counts (100m resolution) dataset](https://hub.worldpop.org/geodata/summary?id=75983) (see attribution and licensing at the end of this notebook), to properly weight each intersecting S2 "sliver" based on actual human density.

*🌟 Note on Temporal Alignment: We explicitly align our 2025 PDI Embeddings with the 2025 WorldPop demographic dataset to avoid temporal confounding, a critical best practice in spatial machine learning!*

### Key Technologies Used

*   **[Population Dynamics Insights](https://developers.google.com/maps/documentation/population-dynamics-insights/overview):** To provide the underlying 330-dimensional embeddings capturing geographic, environmental, and map features.
*   **[BigQuery GIS](https://cloud.google.com/bigquery):** To execute spatial overlays (`ST_INTERSECTS`, `ST_INTERSECTION`) and math natively in the data warehouse.
*   **[Google Earth Engine (GEE)](https://earthengine.google.com/):** Used natively within BigQuery via `ST_REGIONSTATS` to access the [WorldPop 2025 Population Counts dataset](https://hub.worldpop.org/geodata/summary?id=75983) without any raster processing overhead.
*   **[CARTO Analytics Toolbox](https://docs.carto.com/data-and-analysis/analytics-toolbox-overview):** To dynamically generate S2 boundary polygons from PDI string tokens.
*   **Python Libraries:** **[Pandas](https://pandas.pydata.org/)** (data manipulation) and **[NumPy](https://numpy.org/)** (mathematical validation).

*Note: This notebook executes queries that incur BigQuery costs. See [BigQuery Pricing](https://cloud.google.com/bigquery/pricing) for details.*

### The Step-by-Step Workflow

1.  **Generate Target Boundaries:** We use BigQuery GIS to construct dummy 5km store isochrones around major US cities to act as our custom boundaries.
2.  **Calculate Total Populations:** We use Earth Engine to compute the total human population residing inside each 5km boundary.
3.  **Intersect & Calculate Slivers:** We load the PDI data, join the S2 cells against our store boundaries, cut out the precise overlapping spatial "slivers", and use Earth Engine again to determine the exact population living within that specific sliver.
4.  **Apply Weights:** We multiply the 330 PDI dimensions by their population weight ratio and sum the vectors natively in BigQuery.
5.  **Optional Python Normalization:** We demonstrate how to apply L2 Renormalization using `scikit-learn` for users whose models require strictly normalized unit vectors.
6.  **Mathematical Validation:** We extract the final array into NumPy to programmatically prove it retains 330 dimensions and a magnitude of `1.0`.

### How to Use This Notebook

1.  **Prerequisites:** Before running this notebook, you must:
    *   Enable the **BigQuery API** and the **Google Earth Engine API** in your Google Cloud Project.
    *   **Project Authentication:**
        *   **Standard Colab Users:** Configure an environment variable in the Colab "Secrets" tab (the **key icon** on the left menu) named `GCP_PROJECT_ID`. This should be your Google Cloud Project ID.
        *   **Colab Enterprise / Vertex AI Users:** No secret configuration is needed. The notebook will automatically authenticate using the active Google Cloud Project you are working in.
    *   **Crucially, the Google Cloud Project you use must have an active subscription/authorization to access the US Population Dynamics Insights dataset.**
2.  **Authentication:** The first code cell will prompt you to authenticate your Google Account. Ensure the account you use has BigQuery Data Viewer/Job User permissions and Earth Engine Resource Viewer permissions for your Project ID.
3.  **Run the Cells:** Once authenticated, execute the cells in order from top to bottom.

In [ ]:
# @title Install Dependencies - Not required in Google Colab Environment
# @markdown Install necessary BigQuery to Pandas dependencies.
# !pip install -q db-dtypes

In [ ]:
# @title 1. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
import sys
import pandas as pd
import numpy as np
import warnings
from google.cloud import bigquery
import google.auth

warnings.filterwarnings('ignore') # Suppress warnings for a cleaner output

try:
    # --- ATTEMPT 1: Standard Consumer Colab ---
    from google.colab import userdata, auth
    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID')
    print("✅ Standard Colab Secrets found.")

    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)

    client = bigquery.Client(project=GCP_PROJECT_ID)
    print(f"✅ Authenticated to Google Cloud Project: {GCP_PROJECT_ID}")

except (ImportError, Exception) as e:
    # --- ATTEMPT 2: Colab Enterprise / Local Jupyter ---
    print(f"ℹ️ Standard Colab setup skipped ({e}). Falling back to Enterprise/Local Auth...")

    credentials, GCP_PROJECT_ID = google.auth.default()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

print("✅ BigQuery Client Initialized.")

### Step 2: Generate Dummy Target Boundaries

Before running the core aggregation, we need custom polygons to act as our target geometries. This BigQuery SQL cell will create a new dataset (`pdi_testing`) and a new table (`store_isochrones`) directly in your Google Cloud Project, populated with five dummy "store trade areas" (5km circular buffers around major US cities) to test the pipeline.

In [ ]:
%%bigquery --project $GCP_PROJECT_ID
CREATE SCHEMA IF NOT EXISTS pdi_testing;

CREATE OR REPLACE TABLE pdi_testing.store_isochrones AS
SELECT 'store_1_ny' AS store_id, ST_BUFFER(ST_GEOGPOINT(-74.0060, 40.7128), 5000) AS isochrone_geom UNION ALL
SELECT 'store_2_la', ST_BUFFER(ST_GEOGPOINT(-118.2437, 34.0522), 5000) UNION ALL
SELECT 'store_3_chi', ST_BUFFER(ST_GEOGPOINT(-87.6298, 41.8781), 5000) UNION ALL
SELECT 'store_4_hou', ST_BUFFER(ST_GEOGPOINT(-95.3698, 29.7604), 5000) UNION ALL
SELECT 'store_5_pho', ST_BUFFER(ST_GEOGPOINT(-112.0740, 33.4484), 5000);

### Step 3: The Aggregation Pipeline (Core Logic)

This cell executes the spatial math and aggregation entirely within BigQuery, saving the results directly to a Pandas DataFrame named `df_aggregated`.

To efficiently and accurately roll up the S2 cells into our custom boundaries, we:
1. Use BigQuery's native `S2_COVERINGCELLIDS` to perform a highly-optimized index match, retrieving only the specific PDI S2 cells that touch our custom polygons.
2. Cut out the precise intersecting "slivers" and use **Google Earth Engine** to calculate the high-resolution WorldPop population inside each sliver.
3. Calculate the weight ratio natively using a BigQuery Window Function, multiply the 330 PDI dimensions by this weight, and sum the vectors to capture the exact density and pattern of the geographic features.

> **💡 Note:** In the SQL cell below, please verify the `JOIN` statement in the `Matched_PDI` CTE references the correct dataset name for your PDI subscription. Depending on how your access was provisioned, you may need to update the table path (e.g., replacing `population_dynamics___us.v1_s2_z12` with your linked dataset name).

In [ ]:
%%bigquery df_aggregated --project $GCP_PROJECT_ID
-- 1. Define UDF to convert BigQuery's S2 INT64 to PDI's Hex String
CREATE TEMP FUNCTION TO_S2_HEX(s2_int INT64) RETURNS STRING AS (
  RTRIM(FORMAT('%x%015x', (s2_int >> 60) & 0xF, s2_int & 0x0FFFFFFFFFFFFFFF), '0')
);

WITH Custom_Boundaries AS (
  -- 2. Load Custom Geometries
  SELECT
    store_id,
    isochrone_geom AS target_geom
  FROM `pdi_testing.store_isochrones`
),

Covering_Cells AS (
  -- 3. Get S2 Level 12 cells that cover our polygon
  -- We use named arguments and bump max_cells to ensure full coverage.
  SELECT
    store_id,
    target_geom,
    TO_S2_HEX(cell_id) AS s2_hex
  FROM Custom_Boundaries,
  UNNEST(S2_COVERINGCELLIDS(target_geom, min_level => 12, max_level => 12, max_cells => 500)) AS cell_id
),

Matched_PDI AS (
  -- 4. Fast String Join: Only generate geometries for the cells we actually need!
  SELECT
    c.store_id,
    c.target_geom,
    p.geo_id AS s2_id,
    p.features,
    `carto-os.carto.S2_BOUNDARY`(`carto-os.carto.S2_FROMTOKEN`(p.geo_id)) AS s2_geom
  FROM Covering_Cells c
  JOIN `population_dynamics___us.v1_s2_z12` p
    ON c.s2_hex = p.geo_id
),

Intersecting_Cells AS (
  -- 5. Cut out the precise intersecting 'sliver' geometry
  SELECT
    store_id,
    s2_id,
    features,
    ST_INTERSECTION(target_geom, s2_geom) AS sliver_geom
  FROM Matched_PDI
  -- Double check intersection since S2 covering is a slight over-approximation
  WHERE ST_INTERSECTS(target_geom, s2_geom)
),

Sliver_Populations AS (
  -- 6. Call Earth Engine to get the population of the sliver
  SELECT
    store_id,
    features,
    ST_REGIONSTATS(
      sliver_geom,
      'ee://projects/sat-io/open-datasets/WORLDPOP/pop/USA_POP_2025_CN_100M_R2025A_V1'
    ).sum AS sliver_pop
  FROM Intersecting_Cells
),

Sliver_Weights AS (
  -- 7. Calculate the weight using a BigQuery Window Function
  SELECT
    store_id,
    features,
    sliver_pop / NULLIF(SUM(sliver_pop) OVER(PARTITION BY store_id), 0) AS weight
  FROM Sliver_Populations
  -- Don't unnest 330 dimensions if the population is 0
  WHERE sliver_pop > 0
),

Weighted_Features AS (
  -- 8. Unnest the arrays, apply the weight, and sum the vectors
  SELECT
    store_id,
    dimension_index,
    SUM(feature_value * weight) AS weighted_sum_value
  FROM Sliver_Weights
  CROSS JOIN UNNEST(features) AS feature_value WITH OFFSET AS dimension_index
  GROUP BY 1, 2
)

-- 9. Final Output: Reconstruct the ML array (Unnormalized)
SELECT
  store_id,
  ARRAY_AGG(
    weighted_sum_value
    ORDER BY dimension_index
  ) AS aggregated_features
FROM Weighted_Features
GROUP BY 1

In [ ]:
# @title 4. Tabular Output
# @markdown View the aggregated DataFrame showing the custom boundaries and their native embeddings.
# @markdown
# @markdown **What you are looking at:**
# @markdown The DataFrame below is the direct result of our BigQuery zero-ETL aggregation. For each `store_id` (representing our custom 5km boundaries), you will see a single `aggregated_features` column.
# @markdown
# @markdown This column contains a 330-dimensional array of floats. Because we did not apply any arbitrary normalization, the magnitude (length) of this vector natively represents the raw volume and signal density of the underlying geographic area.

# Validate DataFrame Shape
print(f"Total Rows Aggregated: {df_aggregated.shape[0]}")
print("Preview of resulting data:")

# Display the clean DataFrame
display(df_aggregated.head())

### Step 5: Optional L2 Normalization in Python

If your downstream use case requires unit vectors (e.g., using Dot Product as a proxy for Cosine Similarity, or performing K-Means clustering based on feature profiles rather than total magnitude), you can apply L2 normalization in Python using Scikit-Learn:

In [ ]:
# @title 5. Apply L2 Normalization
from sklearn.preprocessing import normalize

if not df_aggregated.empty:
    # 1. Extract the aggregated features into a 2D NumPy array
    X_matrix = np.stack(df_aggregated['aggregated_features'].values)

    # 2. Apply L2 normalization
    X_normalized = normalize(X_matrix, norm='l2')

    print(f"Original matrix shape: {X_matrix.shape}")
    print(f"Normalized matrix shape: {X_normalized.shape}")
    print("✅ Normalization complete.")
else:
    print("⚠️ The DataFrame is empty. Ensure the BigQuery aggregation cell ran successfully.")

In [ ]:
# @title 6. Mathematical Validation
# @markdown Verify that the output vector has the correct PDI dimensions (330) and that the Scikit-Learn L2 Renormalization successfully resulted in a unit vector.
# @markdown
# @markdown **Why do we run this test?**
# @markdown Before passing aggregated spatial data into distance-based machine learning algorithms or Vector Databases, we want to mathematically prove our Python pipeline correctly formatted the embedding space.
# @markdown
# @markdown This cell takes the first result from our normalized array and uses Python's `numpy` library to test two strict conditions:
# @markdown 1. **Dimensionality:** Does the resulting array still have exactly 330 dimensions?
# @markdown 2. **Euclidean Length (L2 Norm):** Does the vector have a magnitude of exactly `1.0`? (Validating our Euclidean distance math: $Length = \sqrt{p_1^2 + p_2^2 + \cdots + p_n^2}$).

if 'X_normalized' in locals():
    # Extract the first vector from our normalized matrix
    first_vector = X_normalized[0]

    # --- 1. Validate Dimensions ---
    num_dimensions = len(first_vector)
    print(f"Vector Dimensions: {num_dimensions}")

    if num_dimensions == 330:
        print("✅ Success: The vector has exactly 330 dimensions, matching the PDI specification.")
    else:
        print(f"❌ Error: Expected 330 dimensions, found {num_dimensions}.")

    # --- 2. Validate L2 Renormalization (Unit Vector) ---
    # np.linalg.norm calculates the Euclidean length of the array
    vector_magnitude = np.linalg.norm(first_vector)
    print(f"\nL2 Norm (Magnitude): {vector_magnitude:.6f}")

    # We use np.isclose to account for extremely minor floating-point math differences
    if np.isclose(vector_magnitude, 1.0, atol=1e-5):
        print("✅ Success: The vector magnitude is ~1.0. L2 Renormalization was applied!")
    else:
        print("❌ Error: The vector is not a unit vector. Renormalization failed.")

else:
    print("⚠️ The X_normalized array does not exist. Ensure Step 5 ran successfully.")

### Data Attribution & Licensing

**[WorldPop Dataset](https://www.worldpop.org/):**
This notebook utilizes high-resolution population data to perform population-weighted spatial aggregations. The underlying raster dataset is sourced from WorldPop.

*   **License:** This data is distributed under the [Creative Commons Attribution 4.0 International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/). Note that WorldPop datasets derived from OpenStreetMap or Microsoft Building Footprints are subject to the Open Database License (ODbL); if you alter or build upon that specific data, the results must be distributed under the same ODbL license.
*   **Recommended Citation:** Bondarenko M., Priyatikanto R., Tejedor-Garavito N., Zhang W., McKeen T., Cunningham A., Woods T., Hilton J., Cihan D., Nosatiuk B., Brinkhoff T., Tatem A., Sorichetta A.. 2025 Constrained estimates of 2015-2030 total number of people per grid square at a resolution of 3 arc (approximately 100m at the equator) R2025A version v1. Global Demographic Data Project - Funded by The Bill and Melinda Gates Foundation (INV-045237). WorldPop - School of Geography and Environmental Science, University of Southampton. DOI:10.5258/SOTON/WP00839